# Sujet 3 — Amélioration de la réussite des étudiants

**Objectif :** analyser les données des étudiants afin de comprendre les profils, identifier les facteurs qui influencent la réussite académique et proposer des recommandations pédagogiques.

Dataset Kaggle/UCI : `student-alcohol-consumption`

> À placer dans le même dossier que ce notebook : `student-mat.csv` ou `student-por.csv`.

## 1. Importation des bibliothèques

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, ConfusionMatrixDisplay, classification_report,
    roc_auc_score, RocCurveDisplay
)

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

## 2. Chargement des données

Le fichier peut être `student-mat.csv` ou `student-por.csv`. Le séparateur est généralement `;`.

In [ ]:
file_path = "student-mat.csv"  # changer en "student-por.csv" si nécessaire

df = pd.read_csv(file_path, sep=";")
print("Dimensions du dataset :", df.shape)
df.head()

## 3. Compréhension initiale du dataset

In [ ]:
print("Informations générales :")
print(df.info())

print("Valeurs manquantes :")
print(df.isna().sum())

print("Nombre de doublons :", df.duplicated().sum())

df.describe(include="all").T

## 4. Création de la variable cible

Dans ce projet, on considère qu’un étudiant réussit si sa note finale `G3` est supérieure ou égale à 10.

- `Success = 1` : réussite
- `Success = 0` : difficulté / échec

In [ ]:
data = df.copy()

data["Success"] = (data["G3"] >= 10).astype(int)

print(data["Success"].value_counts())
print("Pourcentage :")
print((data["Success"].value_counts(normalize=True) * 100).round(2))

## 5. Analyse exploratoire des données

In [ ]:
success_counts = data["Success"].value_counts().sort_index()

plt.figure(figsize=(6,4))
success_counts.plot(kind="bar")
plt.title("Répartition réussite / difficulté")
plt.xlabel("Success (0 = difficulté, 1 = réussite)")
plt.ylabel("Nombre d'étudiants")
plt.xticks(rotation=0)
plt.show()

In [ ]:
plt.figure(figsize=(7,4))
data["G3"].hist(bins=15)
plt.title("Distribution de la note finale G3")
plt.xlabel("Note finale")
plt.ylabel("Nombre d'étudiants")
plt.show()

In [ ]:
plt.figure(figsize=(7,4))
data.groupby("studytime")["G3"].mean().plot(kind="bar")
plt.title("Note finale moyenne selon le temps d'étude")
plt.xlabel("Temps d'étude")
plt.ylabel("Moyenne G3")
plt.xticks(rotation=0)
plt.show()

In [ ]:
plt.figure(figsize=(7,4))
data.groupby("failures")["G3"].mean().plot(kind="bar")
plt.title("Note finale moyenne selon les échecs précédents")
plt.xlabel("Nombre d'échecs précédents")
plt.ylabel("Moyenne G3")
plt.xticks(rotation=0)
plt.show()

In [ ]:
plt.figure(figsize=(7,4))
data.groupby("absences")["G3"].mean().plot(kind="line")
plt.title("Relation entre absences et note finale")
plt.xlabel("Absences")
plt.ylabel("Moyenne G3")
plt.show()

In [ ]:
if "Dalc" in data.columns and "Walc" in data.columns:
    data["AlcoholMean"] = (data["Dalc"] + data["Walc"]) / 2

    plt.figure(figsize=(7,4))
    data.groupby("AlcoholMean")["G3"].mean().plot(kind="bar")
    plt.title("Note finale moyenne selon la consommation d'alcool")
    plt.xlabel("Niveau moyen d'alcool")
    plt.ylabel("Moyenne G3")
    plt.xticks(rotation=0)
    plt.show()

## 6. Préparation des données pour la modélisation

On retire `G1`, `G2` et `G3` pour éviter que le modèle n’utilise directement les notes déjà connues.

In [ ]:
target = "Success"

cols_to_drop = ["G1", "G2", "G3", target]
X = data.drop(columns=[col for col in cols_to_drop if col in data.columns])
y = data[target]

categorical_cols = X.select_dtypes(include=["object"]).columns.tolist()
numeric_cols = X.select_dtypes(include=np.number).columns.tolist()

print("Colonnes catégorielles :", categorical_cols)
print("Colonnes numériques :", numeric_cols)

In [ ]:
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, numeric_cols),
    ("cat", categorical_transformer, categorical_cols)
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Taille X_train :", X_train.shape)
print("Taille X_test  :", X_test.shape)

## 7. Entraînement et comparaison des modèles

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=2000, class_weight="balanced"),
    "Decision Tree": DecisionTreeClassifier(random_state=42, class_weight="balanced"),
    "Random Forest": RandomForestClassifier(n_estimators=300, random_state=42, class_weight="balanced")
}

results = []
fitted_pipelines = {}

for name, model in models.items():
    pipeline = Pipeline(steps=[("preprocessor", preprocessor), ("model", model)])
    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)
    if hasattr(pipeline.named_steps["model"], "predict_proba"):
        y_prob = pipeline.predict_proba(X_test)[:, 1]
        auc = roc_auc_score(y_test, y_prob)
    else:
        auc = np.nan
    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1-score": f1_score(y_test, y_pred),
        "ROC-AUC": auc
    })
    fitted_pipelines[name] = pipeline

results_df = pd.DataFrame(results).sort_values(by="F1-score", ascending=False)
results_df

## 8. Évaluation détaillée du meilleur modèle

In [ ]:
best_model_name = results_df.iloc[0]["Model"]
best_pipeline = fitted_pipelines[best_model_name]

print("Meilleur modèle selon le F1-score :", best_model_name)

y_pred_best = best_pipeline.predict(X_test)
print("Classification report :")
print(classification_report(y_test, y_pred_best))

In [ ]:
cm = confusion_matrix(y_test, y_pred_best)
disp = ConfusionMatrixDisplay(confusion_matrix=cm)
disp.plot()
plt.title(f"Matrice de confusion - {best_model_name}")
plt.show()

In [ ]:
if hasattr(best_pipeline.named_steps["model"], "predict_proba"):
    y_prob_best = best_pipeline.predict_proba(X_test)[:, 1]
    RocCurveDisplay.from_predictions(y_test, y_prob_best)
    plt.title(f"Courbe ROC - {best_model_name}")
    plt.show()

## 9. Importance des variables

In [ ]:
model = best_pipeline.named_steps["model"]
preprocessor_fitted = best_pipeline.named_steps["preprocessor"]
feature_names = preprocessor_fitted.get_feature_names_out()

if hasattr(model, "feature_importances_"):
    importances = model.feature_importances_
elif hasattr(model, "coef_"):
    importances = np.abs(model.coef_[0])
else:
    importances = None

if importances is not None:
    importance_df = pd.DataFrame({"Feature": feature_names, "Importance": importances}).sort_values(by="Importance", ascending=False)
    display(importance_df.head(15))
    plt.figure(figsize=(10,6))
    plt.barh(importance_df["Feature"].head(15)[::-1], importance_df["Importance"].head(15)[::-1])
    plt.title(f"Top 15 des variables importantes - {best_model_name}")
    plt.xlabel("Importance")
    plt.tight_layout()
    plt.show()
else:
    print("Impossible d'extraire directement l'importance des variables.")

## 10. Segmentation des profils étudiants

In [ ]:
profile_features = [col for col in ["studytime", "failures", "absences", "Dalc", "Walc", "goout", "health"] if col in data.columns]
profile_data = data[profile_features].copy()

scaler = StandardScaler()
profile_scaled = scaler.fit_transform(profile_data)

inertias = []
K_range = range(2, 9)
for k in K_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(profile_scaled)
    inertias.append(kmeans.inertia_)

plt.figure(figsize=(7,4))
plt.plot(list(K_range), inertias, marker="o")
plt.title("Méthode du coude pour choisir K")
plt.xlabel("Nombre de clusters")
plt.ylabel("Inertie")
plt.show()

In [ ]:
k = 3
kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
data["Cluster"] = kmeans.fit_predict(profile_scaled)

cluster_profile = data.groupby("Cluster")[profile_features + ["G3", "Success"]].mean().round(2)
cluster_size = data["Cluster"].value_counts().sort_index()

print("Taille des clusters :")
print(cluster_size)
display(cluster_profile)

In [ ]:
pca = PCA(n_components=2)
components = pca.fit_transform(profile_scaled)

plt.figure(figsize=(7,5))
plt.scatter(components[:,0], components[:,1], c=data["Cluster"])
plt.title("Visualisation des profils étudiants par cluster")
plt.xlabel("Composante principale 1")
plt.ylabel("Composante principale 2")
plt.show()

## 11. Interprétation métier et recommandations pédagogiques

Exemples de recommandations possibles :

- Étudiants avec faible temps d'étude : accompagnement méthodologique.
- Étudiants avec beaucoup d'absences : suivi administratif et pédagogique.
- Étudiants avec échecs précédents : tutorat personnalisé.
- Étudiants avec habitudes de sortie ou alcool élevées : sensibilisation et soutien.
- Étudiants à fort risque d'échec : détection précoce et plan d'action individuel.

## 12. Conclusion

Cette démarche permet de comprendre les profils des étudiants, identifier les facteurs influençant la réussite, prédire les étudiants à risque et proposer des recommandations pédagogiques adaptées.